# 02. Full Fine-tuning

## 학습 목표
- Instruction Fine-tuning의 필요성과 원리 이해
- 데이터 포맷 (Alpaca, ChatML) 이해
- HuggingFace Trainer로 소규모 모델 Fine-tuning 실습
- Full Fine-tuning의 한계 인식

## 참고 자료
- [Alpaca 논문 (Taori et al., 2023)](https://crfm.stanford.edu/2023/03/13/alpaca.html)
- [HuggingFace Trainer 문서](https://huggingface.co/docs/transformers/main_classes/trainer)

---

In [ ]:
# Google Colab 환경 설정
!pip install -q transformers datasets accelerate

In [ ]:
import torch
from transformers import (
    AutoModelForCausalLM, AutoTokenizer,
    TrainingArguments, Trainer, DataCollatorForLanguageModeling
)
from datasets import load_dataset, Dataset
import matplotlib.pyplot as plt
import json
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Instruction Fine-tuning: 왜 base 모델은 지시를 못 따르는가

### Base 모델 vs Instruction-tuned 모델

Base 모델 (GPT-2, LLaMA-base 등)은 **다음 토큰 예측**만 학습했다:

$$P(x_t | x_1, x_2, \dots, x_{t-1})$$

그래서 "한국의 수도는?"이라는 질문에 대해:

| 모델 | 출력 |
|------|------|
| Base 모델 | "한국의 수도는? 이 질문은 초등학교에서..." (계속 이어 씀) |
| Instruction-tuned | "서울입니다." |

Base 모델은 인터넷 텍스트의 패턴을 학습했으므로, **질문에 답하는 대신 문맥을 이어쓰려고 한다**.

Instruction Fine-tuning은 모델에게 **"질문이 들어오면 답을 생성하라"**는 행동 패턴을 가르치는 과정이다.

In [ ]:
# Base 모델의 행동 확인
model_name = 'gpt2'
tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

# pad_token 설정 (GPT-2는 기본적으로 없음)
tokenizer.pad_token = tokenizer.eos_token

def generate_text(model, prompt, max_new_tokens=100):
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Base 모델에 지시를 내려보기
prompts = [
    "Translate the following English to French: Hello, how are you?",
    "Summarize the following text: Machine learning is a subset of artificial intelligence.",
    "Answer the question: What is the capital of France?"
]

print("Base GPT-2의 출력 (Instruction을 따르지 못함):")
print("=" * 60)
for prompt in prompts:
    output = generate_text(base_model, prompt, max_new_tokens=50)
    print(f"\nPrompt: {prompt}")
    print(f"Output: {output}")
    print("-" * 60)

---
## 2. 데이터 포맷: Alpaca format, ChatML

### Alpaca Format

Stanford Alpaca에서 정의한 instruction 데이터 포맷:

```json
{
    "instruction": "Translate the following to French.",
    "input": "Hello, how are you?",
    "output": "Bonjour, comment allez-vous?"
}
```

학습 시에는 아래와 같은 프롬프트 템플릿으로 변환:

```
Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Input:
{input}

### Response:
{output}
```

### ChatML Format

OpenAI에서 사용하는 대화 포맷:

```
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
What is the capital of France?<|im_end|>
<|im_start|>assistant
The capital of France is Paris.<|im_end|>
```

In [ ]:
# Alpaca 포맷 프롬프트 템플릿
ALPACA_PROMPT_WITH_INPUT = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Input:
{input}

### Response:
{output}"""

ALPACA_PROMPT_NO_INPUT = """Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Response:
{output}"""

def format_alpaca(example):
    """Alpaca 데이터를 프롬프트 템플릿에 맞게 변환"""
    if example.get('input', '') and example['input'].strip():
        text = ALPACA_PROMPT_WITH_INPUT.format(
            instruction=example['instruction'],
            input=example['input'],
            output=example['output']
        )
    else:
        text = ALPACA_PROMPT_NO_INPUT.format(
            instruction=example['instruction'],
            output=example['output']
        )
    return {'text': text}

# 예시 데이터
example_with_input = {
    'instruction': 'Translate the following to French.',
    'input': 'Hello, how are you?',
    'output': 'Bonjour, comment allez-vous?'
}

example_no_input = {
    'instruction': 'What is the capital of Japan?',
    'input': '',
    'output': 'The capital of Japan is Tokyo.'
}

print("=== With Input ===")
print(format_alpaca(example_with_input)['text'])
print()
print("=== Without Input ===")
print(format_alpaca(example_no_input)['text'])

---
## 3. HuggingFace Trainer 설정: TrainingArguments 주요 파라미터

| 파라미터 | 의미 | 권장값 |
|----------|------|--------|
| `learning_rate` | 학습률 | 2e-5 ~ 5e-5 |
| `num_train_epochs` | 에폭 수 | 1~3 (LLM은 보통 1~2) |
| `per_device_train_batch_size` | GPU당 배치 크기 | 메모리에 맞게 (4~16) |
| `gradient_accumulation_steps` | 그래디언트 누적 | effective batch = batch_size * accumulation |
| `warmup_ratio` | 학습률 워밍업 비율 | 0.03~0.1 |
| `weight_decay` | L2 정규화 | 0.01~0.1 |
| `fp16` / `bf16` | 혼합 정밀도 학습 | GPU 지원 시 True |
| `logging_steps` | 로그 간격 | 10~50 |
| `save_strategy` | 체크포인트 저장 | 'epoch' 또는 'steps' |
| `lr_scheduler_type` | 학습률 스케줄러 | 'cosine' 또는 'linear' |

### Effective Batch Size 계산

$$\text{Effective Batch Size} = \text{per\_device\_batch} \times \text{num\_gpus} \times \text{gradient\_accumulation}$$

예: batch_size=4, 1 GPU, accumulation=8 → Effective Batch Size = 32

In [ ]:
# TrainingArguments 예시
training_args = TrainingArguments(
    output_dir='./results',
    
    # 학습 하이퍼파라미터
    num_train_epochs=3,
    learning_rate=5e-5,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,  # effective batch = 4 * 4 = 16
    
    # 학습률 스케줄러
    warmup_ratio=0.03,
    lr_scheduler_type='cosine',
    
    # 정규화
    weight_decay=0.01,
    
    # 혼합 정밀도 (Colab T4에서 fp16 사용)
    fp16=torch.cuda.is_available(),
    
    # 로깅
    logging_steps=10,
    logging_dir='./logs',
    
    # 체크포인트
    save_strategy='epoch',
    save_total_limit=2,
    
    # 기타
    report_to='none',  # wandb 등 비활성화
    remove_unused_columns=False,
)

print("TrainingArguments 설정 완료!")
print(f"  Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  FP16: {training_args.fp16}")

---
## 4. 실습: GPT-2 Instruction Fine-tuning

소규모 모델(GPT-2, 124M 파라미터)을 사용하여 전체 과정을 실습한다.

```
데이터 준비 → 토큰화 → Trainer 설정 → 학습 → 생성 테스트
```

In [ ]:
# Step 1: 학습 데이터 준비 (소규모 instruction 데이터)
# 실제 프로젝트에서는 Alpaca, Dolly 등의 데이터셋을 사용
# 여기서는 학습 시간을 위해 소규모 데이터를 직접 만든다

instruction_data = [
    {"instruction": "What is machine learning?", "input": "",
     "output": "Machine learning is a branch of artificial intelligence that enables computers to learn patterns from data without being explicitly programmed."},
    {"instruction": "Explain the concept of overfitting.", "input": "",
     "output": "Overfitting occurs when a model learns the training data too well, including noise and outliers, resulting in poor performance on new, unseen data."},
    {"instruction": "What is the difference between supervised and unsupervised learning?", "input": "",
     "output": "Supervised learning uses labeled data to train models for prediction, while unsupervised learning finds patterns in unlabeled data without predefined outputs."},
    {"instruction": "Summarize the following text.", "input": "Neural networks are computing systems inspired by biological neural networks. They consist of layers of interconnected nodes that process information.",
     "output": "Neural networks are bio-inspired computing systems made of layered, interconnected nodes for information processing."},
    {"instruction": "Translate to French.", "input": "Hello, how are you?",
     "output": "Bonjour, comment allez-vous?"},
    {"instruction": "What is gradient descent?", "input": "",
     "output": "Gradient descent is an optimization algorithm that iteratively adjusts model parameters by moving in the direction of steepest decrease of the loss function."},
    {"instruction": "Explain what a transformer architecture is.", "input": "",
     "output": "A transformer is a neural network architecture that uses self-attention mechanisms to process sequential data in parallel, enabling better capture of long-range dependencies."},
    {"instruction": "What is transfer learning?", "input": "",
     "output": "Transfer learning is a technique where a model trained on one task is adapted for a different but related task, leveraging previously learned knowledge."},
    {"instruction": "List three common activation functions.", "input": "",
     "output": "Three common activation functions are: 1) ReLU (Rectified Linear Unit), 2) Sigmoid, and 3) Tanh (Hyperbolic Tangent)."},
    {"instruction": "What is the purpose of dropout in neural networks?", "input": "",
     "output": "Dropout is a regularization technique that randomly deactivates neurons during training to prevent overfitting and improve model generalization."},
    {"instruction": "Explain backpropagation.", "input": "",
     "output": "Backpropagation is an algorithm for training neural networks that computes gradients of the loss function with respect to each weight by applying the chain rule from output to input layers."},
    {"instruction": "What is a learning rate?", "input": "",
     "output": "A learning rate is a hyperparameter that controls how much to adjust model weights during training. Too high causes instability, too low causes slow convergence."},
    {"instruction": "Classify the sentiment of the following text.", "input": "I absolutely loved this movie! The acting was superb.",
     "output": "Positive sentiment. The text expresses strong enthusiasm and praise."},
    {"instruction": "Classify the sentiment of the following text.", "input": "This was the worst experience ever. Terrible service.",
     "output": "Negative sentiment. The text expresses strong dissatisfaction."},
    {"instruction": "What is batch normalization?", "input": "",
     "output": "Batch normalization is a technique that normalizes layer inputs by adjusting and scaling activations, which helps stabilize and accelerate neural network training."},
    {"instruction": "What is the vanishing gradient problem?", "input": "",
     "output": "The vanishing gradient problem occurs in deep neural networks when gradients become extremely small during backpropagation, causing earlier layers to learn very slowly or not at all."},
]

# Alpaca 포맷으로 변환
formatted_data = [format_alpaca(d) for d in instruction_data]

# 데이터 수를 늘리기 위해 반복 (실습용)
formatted_data = formatted_data * 10  # 160개로 증폭

train_dataset = Dataset.from_list(formatted_data)

print(f"학습 데이터 수: {len(train_dataset)}")
print(f"\n첫 번째 데이터 예시:")
print(train_dataset[0]['text'][:300])

In [ ]:
# Step 2: 토큰화
model_name = 'gpt2'
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

MAX_LENGTH = 256

def tokenize_function(examples):
    """텍스트를 토큰화하고, labels를 input_ids와 동일하게 설정 (Causal LM)"""
    tokenized = tokenizer(
        examples['text'],
        truncation=True,
        max_length=MAX_LENGTH,
        padding='max_length',
    )
    # Causal LM: labels = input_ids (다음 토큰 예측)
    tokenized['labels'] = tokenized['input_ids'].copy()
    return tokenized

tokenized_dataset = train_dataset.map(tokenize_function, batched=True, remove_columns=['text'])
tokenized_dataset.set_format('torch')

print(f"토큰화 완료!")
print(f"Keys: {list(tokenized_dataset[0].keys())}")
print(f"input_ids shape: {tokenized_dataset[0]['input_ids'].shape}")

# 디코딩해서 확인
decoded = tokenizer.decode(tokenized_dataset[0]['input_ids'], skip_special_tokens=True)
print(f"\n디코딩된 텍스트:\n{decoded[:200]}...")

In [ ]:
# Step 3: 모델 로드 + Trainer 설정
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
model.config.pad_token_id = tokenizer.eos_token_id

print(f"모델 파라미터: {sum(p.numel() for p in model.parameters()):,}")

training_args = TrainingArguments(
    output_dir='./gpt2-instruction-ft',
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=5e-5,
    warmup_ratio=0.1,
    lr_scheduler_type='cosine',
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    logging_steps=10,
    save_strategy='epoch',
    save_total_limit=1,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
)

In [ ]:
# Step 4: 학습
print("학습 시작...")
train_result = trainer.train()
print(f"\n학습 완료!")
print(f"Total steps: {train_result.global_step}")
print(f"Training loss: {train_result.training_loss:.4f}")

In [ ]:
# Step 5: 생성 테스트 - Fine-tuning 전후 비교

# Fine-tuned 모델로 생성
ft_model = model
ft_model.eval()

# Base 모델 다시 로드
base_model = AutoModelForCausalLM.from_pretrained('gpt2').to(device)
base_model.eval()

test_prompts = [
    "Below is an instruction that describes a task. Write a response that appropriately completes the request.\n\n### Instruction:\nWhat is deep learning?\n\n### Response:\n",
    "Below is an instruction that describes a task. Write a response that appropriately completes the request.\n\n### Instruction:\nExplain what an epoch means in machine learning.\n\n### Response:\n",
]

print("Base vs Fine-tuned 모델 비교")
print("=" * 70)

for prompt in test_prompts:
    print(f"\nPrompt: {prompt.split('Instruction:')[1].split('Response:')[0].strip()}")
    print("~" * 70)
    
    # Base
    base_output = generate_text(base_model, prompt, max_new_tokens=80)
    base_response = base_output.split('### Response:')[-1].strip()
    print(f"[Base]       {base_response[:200]}")
    
    # Fine-tuned
    ft_output = generate_text(ft_model, prompt, max_new_tokens=80)
    ft_response = ft_output.split('### Response:')[-1].strip()
    print(f"[Fine-tuned] {ft_response[:200]}")
    print("-" * 70)

---
## 5. 학습 모니터링: Loss Curve, 생성 샘플 확인

학습이 제대로 되고 있는지 확인하는 두 가지 방법:
1. **Loss curve**: 학습 손실이 안정적으로 감소하는지
2. **생성 샘플**: 실제 출력 품질을 눈으로 확인

In [ ]:
# Loss curve 시각화
log_history = trainer.state.log_history

# 학습 loss만 추출
train_steps = [entry['step'] for entry in log_history if 'loss' in entry]
train_losses = [entry['loss'] for entry in log_history if 'loss' in entry]

if train_steps:
    plt.figure(figsize=(10, 4))
    plt.plot(train_steps, train_losses, 'b-', alpha=0.7)
    plt.xlabel('Step')
    plt.ylabel('Training Loss')
    plt.title('Training Loss Curve')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print(f"초기 Loss: {train_losses[0]:.4f}")
    print(f"최종 Loss: {train_losses[-1]:.4f}")
    print(f"감소율: {(1 - train_losses[-1]/train_losses[0])*100:.1f}%")
else:
    print("로그가 아직 없습니다. 학습을 먼저 실행하세요.")

---
## 6. Full Fine-tuning의 한계

### 메모리 요구량

Full Fine-tuning 시 GPU 메모리에 저장해야 하는 것:

| 항목 | 메모리 (7B 모델 기준) |
|------|----------------------|
| 모델 파라미터 (fp16) | ~14 GB |
| Gradient (fp16) | ~14 GB |
| Optimizer states (Adam: fp32) | ~56 GB |
| Activations (배치에 따라) | ~10-20 GB |
| **합계** | **~100+ GB** |

→ 7B 모델의 Full Fine-tuning에는 **A100 80GB GPU가 최소 2장** 필요

### Catastrophic Forgetting

특정 태스크에 맞게 Fine-tuning하면 사전학습에서 배운 지식을 **잊어버릴 수 있다**.

$$P_{\text{fine-tuned}}(\text{general knowledge}) \ll P_{\text{base}}(\text{general knowledge})$$

### Full Fine-tuning이 적합한 경우
- 충분한 GPU 리소스가 있을 때
- 태스크별 전용 모델을 만들 때
- 데이터가 충분히 많을 때 (수만 개 이상)

In [ ]:
# Full Fine-tuning 메모리 추정 시각화
model_sizes = ['1B', '3B', '7B', '13B', '70B']
params = [1, 3, 7, 13, 70]

# 메모리 추정 (GB): params * 2 (fp16) + params * 2 (grad) + params * 8 (Adam) + ~10% activations
memory_model = [p * 2 for p in params]         # 모델 (fp16)
memory_grad = [p * 2 for p in params]           # Gradient
memory_optim = [p * 8 for p in params]          # Optimizer (Adam fp32)
memory_act = [p * 2 for p in params]            # Activations (추정)

fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(model_sizes))
width = 0.6

bottom = [0] * len(model_sizes)
colors = ['#2196F3', '#FF9800', '#F44336', '#4CAF50']
labels = ['Model (fp16)', 'Gradients', 'Optimizer (Adam)', 'Activations']
memories = [memory_model, memory_grad, memory_optim, memory_act]

for mem, color, label in zip(memories, colors, labels):
    ax.bar(x, mem, width, bottom=bottom, color=color, label=label)
    bottom = [b + m for b, m in zip(bottom, mem)]

# GPU 메모리 라인
ax.axhline(y=16, color='gray', linestyle='--', alpha=0.7, label='T4 16GB')
ax.axhline(y=24, color='gray', linestyle='-.', alpha=0.7, label='RTX 3090 24GB')
ax.axhline(y=80, color='gray', linestyle=':', alpha=0.7, label='A100 80GB')

ax.set_xticks(x)
ax.set_xticklabels(model_sizes)
ax.set_xlabel('Model Size')
ax.set_ylabel('GPU Memory (GB)')
ax.set_title('Full Fine-tuning Memory Requirements')
ax.legend(loc='upper left', fontsize=9)
ax.set_ylim(0, 1100)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\n모델 크기별 Full Fine-tuning 메모리 요구량 (추정):")
for size, total in zip(model_sizes, bottom):
    print(f"  {size:>4s}: ~{total:>5.0f} GB")

---
## 연습 문제

아래 문제를 직접 풀어보세요.

### 연습 1: 한국어 Instruction 데이터로 Fine-tuning

아래 한국어 instruction 데이터를 사용하여 GPT-2를 Fine-tuning하세요.
- Alpaca 포맷으로 데이터를 변환
- Trainer로 학습 (2~3 에폭)
- 학습 전후 생성 결과 비교

In [ ]:
# TODO: 한국어 Instruction Fine-tuning
# Hint:
# 1. 한국어 instruction 데이터 생성 (또는 HuggingFace에서 한국어 데이터셋 로드)
# 2. Alpaca 포맷으로 변환
# 3. 토큰화 + Trainer 설정
# 4. 학습 + 생성 테스트

# 예시 한국어 데이터:
# ko_data = [
#     {"instruction": "인공지능이란 무엇인가요?", "input": "",
#      "output": "인공지능(AI)은 인간의 학습, 추론, 판단 능력을 컴퓨터로 구현하는 기술입니다."},
#     {"instruction": "다음 문장을 요약하세요.", "input": "딥러닝은 인공 신경망을 기반으로...",
#      "output": "딥러닝은 다층 신경망으로 복잡한 패턴을 학습하는 기술입니다."},
# ]
#
# Note: GPT-2는 영어 위주 모델이므로, 한국어 성능이 제한적일 수 있습니다.
# 실제로는 polyglot-ko, kullm 등 한국어 모델을 사용하는 것이 좋습니다.

---
## 핵심 정리

| 개념 | 설명 | 핵심 포인트 |
|------|------|-------------|
| Instruction Fine-tuning | 모델에게 지시 따르기를 가르침 | Base 모델은 지시를 따르지 못함 |
| Alpaca Format | instruction/input/output 구조 | 가장 널리 쓰이는 포맷 |
| ChatML | system/user/assistant 대화 구조 | 대화형 모델에 사용 |
| TrainingArguments | HuggingFace 학습 설정 | lr, batch_size, warmup 등 |
| Loss Monitoring | 학습 곡선 + 생성 샘플 확인 | 수치 + 눈으로 이중 확인 |
| Full FT 한계 | 메모리, 비용, catastrophic forgetting | → PEFT/LoRA로 해결 |

**다음 노트북**: [03-peft-lora.ipynb](03-peft-lora.ipynb) - Parameter-Efficient Fine-Tuning (LoRA, QLoRA)